In [ ]:
import numpy as np
import cv2
import pywt
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

IMAGE_PATH = r"C:\Users\Lenovo\Downloads\archive\GAN-Traning Images\Tr-pi_1366.jpg"
SAMPLING_RATE = 0.7   # same as paper (try 0.35, 0.5, 0.7)


def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (256, 256))
    img = img.astype(np.float64)
    img = (img - img.min()) / (img.max() - img.min())   # mat2gray
    return img


def compute_metrics(original, recovered):
    mse = np.mean((original - recovered) ** 2)
    rmse = np.sqrt(mse)
    psnr_val = psnr(original, recovered, data_range=1.0)
    ssim_val = ssim(original, recovered, data_range=1.0)
    return rmse, psnr_val, ssim_val


def PROCESSIMAGE_DWT(image, samplingRate=0.5):
    print("\nRunning Algorithm 1: DWT")

    signal = image.copy()

 
    coeffs = pywt.wavedec2(signal, 'haar', level=1)
    coeff_array, coeff_slices = pywt.coeffs_to_array(coeffs)

    M = coeff_array.size
    N = int(M * samplingRate)

    indices = np.random.choice(M, N, replace=False)
    sampled_coeffs = np.zeros(M)
    sampled_coeffs[indices] = coeff_array.flatten()[indices]

    sampled_coeffs = sampled_coeffs.reshape(coeff_array.shape)


    coeffs_rec = pywt.array_to_coeffs(sampled_coeffs, coeff_slices, output_format='wavedec2')
    recoveredSignal = pywt.waverec2(coeffs_rec, 'haar')
    recoveredSignal = np.clip(recoveredSignal, 0, 1)


    rmse, psnr_val, ssim_val = compute_metrics(signal, recoveredSignal)

    print("RMSE :", rmse)
    print("PSNR :", psnr_val)
    print("SSIM :", ssim_val)

    return recoveredSignal


def COMPRESSIONFFT(image, samplingRate=0.5):
    print("\nRunning Algorithm 2: FFT")

    signal = image.copy()


    coeffs = np.fft.fft2(signal)


    M = coeffs.size
    N = int(M * samplingRate)

    indices = np.random.choice(M, N, replace=False)
    flat_coeffs = coeffs.flatten()

    recoveredCoeffs = np.zeros_like(flat_coeffs)
    recoveredCoeffs[indices] = flat_coeffs[indices]
    recoveredCoeffs = recoveredCoeffs.reshape(coeffs.shape)


    recoveredSignal = np.real(np.fft.ifft2(recoveredCoeffs))
    recoveredSignal = np.clip(recoveredSignal, 0, 1)

    rmse, psnr_val, ssim_val = compute_metrics(signal, recoveredSignal)

    print("RMSE :", rmse)
    print("PSNR :", psnr_val)
    print("SSIM :", ssim_val)

    return recoveredSignal

def show_results(original, dwt_img, fft_img):
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(original, cmap='gray')
    plt.title("Original MRI")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(dwt_img, cmap='gray')
    plt.title("DWT Reconstruction")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(fft_img, cmap='gray')
    plt.title("FFT Reconstruction")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    image = load_image(IMAGE_PATH)

    dwt_result = PROCESSIMAGE_DWT(image, SAMPLING_RATE)
    fft_result = COMPRESSIONFFT(image, SAMPLING_RATE)

    show_results(image, dwt_result, fft_result)


ModuleNotFoundError: No module named 'pywt'